In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

print(f"TensorFlow Version: {tf.__version__}")
print("Pipeline configurado para predição de anomalias cardíacas.")

TensorFlow Version: 2.20.0


In [ ]:
nome_arquivo = '../datasets/dataset2.csv' 
df = pd.read_csv(nome_arquivo, sep=',') 

df.columns = df.columns.str.replace('#', '').str.strip().str.lower()
df.rename(columns={'heart_rate': 'bpm', 'activityid': 'target'}, inplace=True)

# Remove o ruído
df = df[df['target'] != 'transient activities'].copy()

# ============================================================
# GERAÇÃO DE FEATURES SINTÉTICAS
# Estratégia: gerar dados que não existem no dataset original
# usando correlações conhecidas da literatura médica.
# Isso permite prototipar o pipeline completo.
# Em produção, substituir por dados reais de wearables.
# ============================================================

np.random.seed(42)

# --- 1. PASSOS (steps) ---
# Inferidos a partir do activityID: atividades mais intensas = mais passos/min
activity_to_steps = {
    'lying': (0, 2),         # deitado: ~0 passos
    'sitting': (0, 3),       # sentado: ~0 passos
    'standing': (0, 5),      # em pé parado: ~0-5 passos
    'walking': (80, 120),    # caminhando: 80-120 passos/min
    'running': (140, 200),   # correndo: 140-200 passos/min
    'cycling': (40, 80),     # ciclismo: passos indiretos (pedais)
    'nordic walking': (90, 130),
    'ascending stairs': (60, 100),
    'descending stairs': (60, 100),
    'vacuum cleaning': (20, 60),
    'ironing': (5, 20),
    'rope jumping': (100, 160),
}

def generate_steps(target_label):
    """Gera contagem de passos baseada na atividade."""
    target_str = str(target_label).lower().strip()
    for activity, (low, high) in activity_to_steps.items():
        if activity in target_str:
            return np.random.randint(low, high + 1)
    return np.random.randint(10, 60)  # default

df['steps'] = df['target'].apply(generate_steps)


# --- 2. INTERVALO RR (ritmo cardíaco) ---
# RR interval (ms) ≈ 60000 / BPM + ruído fisiológico
# Variabilidade normal: ±5-15% do valor base
# Arritmia simulada: variabilidade > 20% em ~3% dos registros
df['rr_interval'] = 60000.0 / df['bpm']
rr_noise = np.random.normal(0, df['rr_interval'] * 0.08)  # 8% de ruído
df['rr_interval'] = df['rr_interval'] + rr_noise
df['rr_interval'] = df['rr_interval'].clip(lower=200, upper=2000)  # limites fisiológicos


# --- 3. DADOS DEMOGRÁFICOS SINTÉTICOS ---
# Distribuição realista para população de estudo cardiovascular
# Cada "paciente" no PAMAP2 recebe dados fixos (consistentes por sessão)
n_rows = len(df)

# Idade: distribuição normal centrada em 55 anos (população cardíaca)
df['age'] = np.random.normal(55, 15, n_rows).clip(18, 90).astype(int)

# Sexo: 0 = feminino, 1 = masculino (distribuição ~55% masc em estudos cardíacos)
df['sex'] = np.random.binomial(1, 0.55, n_rows)

# Histórico clínico: flags binárias independentes com prevalências realistas
df['hist_hipertensao'] = np.random.binomial(1, 0.35, n_rows)      # 35% prevalência
df['hist_diabetes'] = np.random.binomial(1, 0.15, n_rows)          # 15% prevalência
df['hist_arritmia'] = np.random.binomial(1, 0.10, n_rows)          # 10% prevalência
df['hist_infarto'] = np.random.binomial(1, 0.05, n_rows)           # 5% prevalência
df['hist_insuficiencia'] = np.random.binomial(1, 0.08, n_rows)     # 8% prevalência


# ============================================================
# TARGET: Redefinição para RISCO CARDÍACO
# 0 = Normal (repouso ou atividade esperada)
# 1 = Anômalo/Risco (padrão cardíaco atípico)
# ============================================================
le = LabelEncoder()
df['target'] = le.fit_transform(df['target']) 

# Critério clínico para anomalia:
# - Repouso (atividades 0,1,2) com BPM dentro do esperado = Normal
# - Atividade intensa com BPM proporcional = Normal
# - BPM desproporcional à atividade = Anômalo
df['target'] = df['target'].apply(lambda x: 0 if x <= 2 else 1)
df.dropna(inplace=True)

print(f"Dados carregados e enriquecidos. Shape: {df.shape}")
print(f"\nColunas disponíveis: {list(df.columns)}")
print(f"\nFeatures sintéticas geradas:")
print(f"  - steps (passos/min): range [{df['steps'].min()}, {df['steps'].max()}]")
print(f"  - rr_interval (ms): range [{df['rr_interval'].min():.0f}, {df['rr_interval'].max():.0f}]")
print(f"  - age: range [{df['age'].min()}, {df['age'].max()}]")
print(f"  - sex: {df['sex'].value_counts().to_dict()}")
print(f"  - hist_hipertensao: {df['hist_hipertensao'].mean():.1%}")
print(f"  - hist_diabetes: {df['hist_diabetes'].mean():.1%}")
print(f"  - hist_arritmia: {df['hist_arritmia'].mean():.1%}")
print(f"  - hist_infarto: {df['hist_infarto'].mean():.1%}")
print(f"  - hist_insuficiencia: {df['hist_insuficiencia'].mean():.1%}")

Dados carregados e limpos. Shape atual: (1936481, 33)


In [ ]:
window_size = 5

# ============================================================
# FEATURE ENGINEERING - MÉTRICAS TEMPORAIS (série temporal)
# ============================================================

# --- BPM (frequência cardíaca) ---
df['bpm_rolling_mean'] = df['bpm'].rolling(window=window_size).mean()
df['bpm_rolling_std'] = df['bpm'].rolling(window=window_size).std()
df['bpm_diff'] = df['bpm'].diff()

# --- PASSOS (atividade física) ---
df['steps_rolling_mean'] = df['steps'].rolling(window=window_size).mean()
# Flag binária: paciente está em atividade física?
STEP_THRESHOLD = 30  # passos/min acima deste valor = atividade
df['is_active'] = (df['steps_rolling_mean'] >= STEP_THRESHOLD).astype(int)

# BPM "corrigido" pela atividade: se ativo, BPM alto é esperado
# Se parado (is_active=0) e BPM alto → sinal de anomalia
# Fórmula: BPM - (contribuição esperada da atividade)
# Em repouso, BPM esperado ~60-80; em atividade, proporcional aos passos
expected_bpm_from_steps = df['steps_rolling_mean'] * 0.5 + 70  # modelo linear simples
df['bpm_excess'] = df['bpm'] - expected_bpm_from_steps  # positivo = BPM acima do esperado

# --- RITMO CARDÍACO (intervalos RR → arritmias) ---
df['rr_rolling_mean'] = df['rr_interval'].rolling(window=window_size).mean()
df['rr_rolling_std'] = df['rr_interval'].rolling(window=window_size).std()

# RMSSD (Root Mean Square of Successive Differences)
# Métrica padrão de HRV (Heart Rate Variability)
# Baixo RMSSD = baixa variabilidade = risco
rr_diff = df['rr_interval'].diff()
df['rr_rmssd'] = (rr_diff ** 2).rolling(window=window_size).mean().apply(np.sqrt)

# Flag de ritmo irregular: coeficiente de variação do RR > 15%
# CV > 15% pode indicar arritmia
df['rr_cv'] = df['rr_rolling_std'] / df['rr_rolling_mean']
ARRHYTHMIA_CV_THRESHOLD = 0.15
df['irregular_rhythm'] = (df['rr_cv'] > ARRHYTHMIA_CV_THRESHOLD).astype(int)

df.dropna(inplace=True)

# ============================================================
# DEFINIÇÃO DOS GRUPOS DE FEATURES
# ============================================================

# Features temporais (série temporal do wearable)
temporal_features = [
    'bpm', 'bpm_rolling_mean', 'bpm_rolling_std', 'bpm_diff',   # frequência cardíaca
    'steps_rolling_mean', 'is_active', 'bpm_excess',             # atividade física
    'rr_rolling_mean', 'rr_rolling_std', 'rr_rmssd', 'rr_cv',   # ritmo cardíaco / HRV
    'irregular_rhythm',                                           # flag de arritmia
]

# Features demográficas (dados do paciente)
demographic_features = ['age', 'sex']

# Features de histórico clínico
clinical_features = [
    'hist_hipertensao', 'hist_diabetes', 'hist_arritmia',
    'hist_infarto', 'hist_insuficiencia',
]

# Features contínuas que precisam de scaling
continuous_features = [
    'bpm', 'bpm_rolling_mean', 'bpm_rolling_std', 'bpm_diff',
    'steps_rolling_mean', 'bpm_excess',
    'rr_rolling_mean', 'rr_rolling_std', 'rr_rmssd', 'rr_cv',
    'age',
]

# Features binárias que NÃO precisam de scaling
binary_features = [
    'is_active', 'irregular_rhythm', 'sex',
    'hist_hipertensao', 'hist_diabetes', 'hist_arritmia',
    'hist_infarto', 'hist_insuficiencia',
]

all_features = temporal_features + demographic_features + clinical_features
y = df['target'].values

print(f"Features temporais ({len(temporal_features)}): {temporal_features}")
print(f"Features demográficas ({len(demographic_features)}): {demographic_features}")
print(f"Features clínicas ({len(clinical_features)}): {clinical_features}")
print(f"Total de features: {len(all_features)}")
print(f"Total de amostras prontas: {len(df)}")

Features extraídas. Total de amostras prontas: 1936477


In [ ]:
# Separação Treino/Teste
X_temporal = df[temporal_features].values
X_demographic = df[demographic_features].values
X_clinical = df[clinical_features].values

X_train_temp, X_test_temp, y_train, y_test = train_test_split(
    X_temporal, y, test_size=0.3, random_state=42, stratify=y
)

# Usar mesmos índices para os outros grupos
X_all = df[all_features].values
X_train_all, X_test_all, _, _ = train_test_split(
    X_all, y, test_size=0.3, random_state=42, stratify=y
)

X_train_demo, X_test_demo, _, _ = train_test_split(
    X_demographic, y, test_size=0.3, random_state=42, stratify=y
)

X_train_clin, X_test_clin, _, _ = train_test_split(
    X_clinical, y, test_size=0.3, random_state=42, stratify=y
)

# Scaling apenas das features contínuas
# Cria índices das features contínuas dentro de cada grupo
temporal_continuous_idx = [i for i, f in enumerate(temporal_features) if f in continuous_features]
temporal_binary_idx = [i for i, f in enumerate(temporal_features) if f in binary_features]
demo_continuous_idx = [i for i, f in enumerate(demographic_features) if f in continuous_features]
demo_binary_idx = [i for i, f in enumerate(demographic_features) if f in binary_features]

# Scaler para features temporais contínuas
scaler_temporal = StandardScaler()
X_train_temp_scaled = X_train_temp.copy()
X_test_temp_scaled = X_test_temp.copy()
X_train_temp_scaled[:, temporal_continuous_idx] = scaler_temporal.fit_transform(X_train_temp[:, temporal_continuous_idx])
X_test_temp_scaled[:, temporal_continuous_idx] = scaler_temporal.transform(X_test_temp[:, temporal_continuous_idx])

# Scaler para features demográficas contínuas (idade)
scaler_demo = StandardScaler()
X_train_demo_scaled = X_train_demo.copy().astype(float)
X_test_demo_scaled = X_test_demo.copy().astype(float)
X_train_demo_scaled[:, demo_continuous_idx] = scaler_demo.fit_transform(X_train_demo[:, demo_continuous_idx].astype(float))
X_test_demo_scaled[:, demo_continuous_idx] = scaler_demo.transform(X_test_demo[:, demo_continuous_idx].astype(float))

# Features clínicas são todas binárias, não precisam de scaling
X_train_clin_scaled = X_train_clin.astype(float)
X_test_clin_scaled = X_test_clin.astype(float)

print("Dados separados e escalonados com sucesso.")
print(f"  Temporal: train={X_train_temp_scaled.shape}, test={X_test_temp_scaled.shape}")
print(f"  Demográfico: train={X_train_demo_scaled.shape}, test={X_test_demo_scaled.shape}")
print(f"  Clínico: train={X_train_clin_scaled.shape}, test={X_test_clin_scaled.shape}")

Dados escalonados com sucesso.


In [ ]:
# ============================================================
# MODELO MULTI-INPUT (Functional API)
# Três branches para diferentes tipos de dados:
#   1. Temporal: dados de série temporal do wearable (BPM, passos, RR)
#   2. Demográfico: idade e sexo do paciente
#   3. Clínico: histórico de condições prévias
# ============================================================

# Branch 1: Features Temporais (BPM + Steps + RR/HRV)
input_temporal = keras.Input(shape=(len(temporal_features),), name='input_temporal')
x_temp = layers.Dense(64, activation='relu', name='temporal_dense1')(input_temporal)
x_temp = layers.Dropout(0.3, name='temporal_dropout')(x_temp)
x_temp = layers.Dense(32, activation='relu', name='temporal_dense2')(x_temp)

# Branch 2: Features Demográficas (Idade, Sexo)
input_demographic = keras.Input(shape=(len(demographic_features),), name='input_demographic')
x_demo = layers.Dense(16, activation='relu', name='demo_dense')(input_demographic)

# Branch 3: Histórico Clínico (5 flags binárias)
input_clinical = keras.Input(shape=(len(clinical_features),), name='input_clinical')
x_clin = layers.Dense(16, activation='relu', name='clinical_dense')(input_clinical)

# Concatenação dos três branches
merged = layers.Concatenate(name='merge')([x_temp, x_demo, x_clin])
x = layers.Dense(64, activation='relu', name='shared_dense1')(merged)
x = layers.Dropout(0.2, name='shared_dropout')(x)
x = layers.Dense(32, activation='relu', name='shared_dense2')(x)

# Saída: probabilidade de anomalia cardíaca (0 a 1)
output = layers.Dense(1, activation='sigmoid', name='output')(x)

model = keras.Model(
    inputs=[input_temporal, input_demographic, input_clinical],
    outputs=output,
    name='cardiac_risk_model'
)

# Compilação
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Recall(name='recall')]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,433 (9.50 KB)

 Trainable params: 2,433 (9.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
class_weight_dict = {0: 3.0, 1: 1.0}

# Treinamento com multi-input: lista de arrays na mesma ordem das inputs do modelo
history = model.fit(
    [X_train_temp_scaled, X_train_demo_scaled, X_train_clin_scaled], 
    y_train, 
    epochs=15, 
    batch_size=32,
    validation_split=0.2, 
    class_weight=class_weight_dict,
    verbose=1
)

Epoch 1/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 54s 2ms/step - accuracy: 0.7943 - loss: 0.5979 - recall: 0.7654 - val_accuracy: 0.7873 - val_loss: 0.4165 - val_recall: 0.7477
Epoch 2/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 47s 1ms/step - accuracy: 0.7966 - loss: 0.5900 - recall: 0.7696 - val_accuracy: 0.7851 - val_loss: 0.4188 - val_recall: 0.7435
Epoch 3/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 67s 2ms/step - accuracy: 0.7975 - loss: 0.5889 - recall: 0.7713 - val_accuracy: 0.7927 - val_loss: 0.4103 - val_recall: 0.7605
Epoch 4/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 74s 2ms/step - accuracy: 0.7984 - loss: 0.5884 - recall: 0.7726 - val_accuracy: 0.7875 - val_loss: 0.4157 - val_recall: 0.7482
Epoch 5/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 63s 2ms/step - accuracy: 0.7976 - loss: 0.5887 - recall: 0.7713 - val_accuracy: 0.8118 - val_loss: 0.4030 - val_recall: 0.7980
Epoch 6/15
33889/33889 ━━━━━━━━━━━━━━━━━━━━ 59s 2ms/step - accuracy: 0.7968 - loss: 0.5893 - recall: 0.7694 - val_accuracy: 0.7832 - val_loss: 0.

In [ ]:
# Predição com multi-input
probs_anomalia = model.predict([X_test_temp_scaled, X_test_demo_scaled, X_test_clin_scaled])

# Converte P(Anomalia) para P(Normal)
probs_normal = 1.0 - probs_anomalia.flatten()

# Limiar de segurança clínica: 
# Se P(Normal) >= 0.8 → classifica como Normal (0)
# Senão → Anômalo / Risco cardíaco (1)
THRESHOLD = 0.8
preds_custom = np.where(probs_normal >= THRESHOLD, 0, 1)

print("\n--- Resultados: Modelo Multi-Input de Risco Cardíaco ---")
print("\nMatriz de Confusão:")
print(confusion_matrix(y_test, preds_custom))

print(f"\nRelatório de Classificação (Threshold = {THRESHOLD}):")
print(classification_report(y_test, preds_custom, target_names=['Normal (0)', 'Anômalo/Risco (1)']))

18155/18155 ━━━━━━━━━━━━━━━━━━━━ 15s 837us/step

--- Resultados Finais da Avaliação ---

Matriz de Confusão:
[[ 79383  61594]
 [ 33608 406359]]

Relatório de Classificação (Threshold = 0.8):
                       precision    recall  f1-score   support

          Repouso (0)       0.70      0.56      0.63    140977
Atividade/Anormal (1)       0.87      0.92      0.90    439967

             accuracy                           0.84    580944
            macro avg       0.79      0.74      0.76    580944
         weighted avg       0.83      0.84      0.83    580944



In [ ]:
print("\n" + "="*60)
print("   AVALIAÇÃO DOS BASELINES DE COMPARAÇÃO")
print("="*60)

# Features concatenadas para modelos clássicos (sklearn)
X_train_flat = np.hstack([X_train_temp_scaled, X_train_demo_scaled, X_train_clin_scaled])
X_test_flat  = np.hstack([X_test_temp_scaled,  X_test_demo_scaled,  X_test_clin_scaled])

def print_baseline_results(name, y_true, y_pred):
    print(f"\n--- {name} ---")
    print("Matriz de Confusão:")
    print(confusion_matrix(y_true, y_pred))
    print("\nRelatório de Classificação:")
    print(classification_report(y_true, y_pred,
                                target_names=['Normal (0)', 'Anômalo/Risco (1)'],
                                zero_division=0))


# ─── Baseline 1: Regressão Logística ────────────────────────────────────────
log_reg = LogisticRegression(class_weight={0: 3.0, 1: 1.0}, random_state=42, max_iter=1000)
log_reg.fit(X_train_flat, y_train)
log_preds = log_reg.predict(X_test_flat)
print_baseline_results("Baseline 1: Regressão Logística (todas as features)", y_test, log_preds)


# ─── Baseline 2: Random Forest ──────────────────────────────────────────────
rf_clf = RandomForestClassifier(
    n_estimators=100, max_depth=10,
    class_weight={0: 3.0, 1: 1.0},
    random_state=42, n_jobs=-1,
)
rf_clf.fit(X_train_flat, y_train)
rf_probs  = rf_clf.predict_proba(X_test_flat)[:, 1]
rf_preds  = np.where((1 - rf_probs) >= THRESHOLD, 0, 1)   # mesmo limiar clínico 0.8
print_baseline_results("Baseline 2: Random Forest (threshold = 0.8)", y_test, rf_preds)


# ─── Baseline 3: Classificador por Regras Clínicas ──────────────────────────
# Simula o raciocínio diagnóstico básico de um profissional de saúde:
#   Sinaliza risco (1) se QUALQUER critério crítico for verdadeiro:
#     • BPM muito alto em repouso  (taquicardia em repouso ≥ 100 bpm)
#     • BPM muito baixo            (bradicardia grave ≤ 40 bpm)
#     • Ritmo irregular detectado  (flag gerado pelo HRV/RMSSD)
#     • BPM excess alto sem atividade
#   Histórico clínico aumenta a sensibilidade do critério
def rule_based_predict(df_test):
    bpm_col        = df_test['bpm']
    is_active_col  = df_test['is_active']
    irr_col        = df_test['irregular_rhythm']
    excess_col     = df_test['bpm_excess']
    hist_cols      = df_test[['hist_arritmia', 'hist_infarto', 'hist_insuficiencia']].max(axis=1)

    # Condições de risco independentes
    taquicardia_repouso = (bpm_col >= 100) & (is_active_col == 0)
    bradicardia_grave   = bpm_col <= 40
    arritmia_detectada  = irr_col == 1
    bpm_desproporcional = excess_col > 30   # BPM > 30bpm acima do esperado pela atividade

    # Histórico de risco reduz o limiar: qualquer flag já sinaliza
    alto_risco_historico = (hist_cols == 1) & (excess_col > 15)

    anomalia = (
        taquicardia_repouso |
        bradicardia_grave   |
        arritmia_detectada  |
        bpm_desproporcional |
        alto_risco_historico
    )
    return anomalia.astype(int).values

# Precisa do DataFrame original do test set para as regras
test_indices = df.index[len(df) - len(y_test):]   # último 30%
# Reconstruir via train_test_split com mesmo random_state
_, df_test_raw, _, _ = train_test_split(df, y, test_size=0.3, random_state=42, stratify=y)

rule_preds = rule_based_predict(df_test_raw)
print_baseline_results("Baseline 3: Regras Clínicas (limiar por domínio)", y_test, rule_preds)


 INICIANDO AVALIAÇÃO DOS BASELINES DE COMPARAÇÃO 

--- Baseline 1: Dummy Classifier (Estratégia: Mais Frequente) ---
Matriz de Confusão:
[[     0 140977]
 [     0 439967]]

Relatório de Classificação:
                       precision    recall  f1-score   support

          Repouso (0)       0.00      0.00      0.00    140977
Atividade/Anormal (1)       0.76      1.00      0.86    439967

             accuracy                           0.76    580944
            macro avg       0.38      0.50      0.43    580944
         weighted avg       0.57      0.76      0.65    580944


--- Baseline 2: Regressão Logística ---
Matriz de Confusão:
[[118366  22611]
 [107457 332510]]

Relatório de Classificação:
                       precision    recall  f1-score   support

          Repouso (0)       0.52      0.84      0.65    140977
Atividade/Anormal (1)       0.94      0.76      0.84    439967

             accuracy                           0.78    580944
            macro avg       0.73      

In [ ]:
print("\n" + "="*60)
print("   ABLATION STUDY — CONTRIBUIÇÃO DE CADA MÉTRICA")
print("="*60)
print("Treina o modelo Logistic Regression com subsets crescentes")
print("de features para mostrar o ganho real de cada adição.\n")

def ablation_eval(X_tr, X_te, y_tr, y_te, label):
    """Treina LogReg no subset e retorna métricas resumidas."""
    clf = LogisticRegression(class_weight={0: 3.0, 1: 1.0}, random_state=42, max_iter=1000)
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    recall_normal   = recall_score(y_te, preds, pos_label=0, zero_division=0)
    recall_anomalia = recall_score(y_te, preds, pos_label=1, zero_division=0)
    f1_macro        = f1_score(y_te, preds, average='macro', zero_division=0)
    acc             = (preds == y_te).mean()
    print(f"  {label:<45} | Acc={acc:.3f} | F1-macro={f1_macro:.3f}"
          f" | Recall-Normal={recall_normal:.3f} | Recall-Risco={recall_anomalia:.3f}")

print(f"  {'Conjunto de Features':<45} | {'Acc':^5}   {'F1-macro':^8}   "
      f"{'Recall-Normal':^13}   {'Recall-Risco':^12}")
print("  " + "-"*100)

# ── Passo 1: apenas BPM (modelo original, 4 features) ───────────────────────
bpm_only_features = ['bpm', 'bpm_rolling_mean', 'bpm_rolling_std', 'bpm_diff']
bpm_idx_train = [temporal_features.index(f) for f in bpm_only_features]
ablation_eval(
    X_train_temp_scaled[:, bpm_idx_train],
    X_test_temp_scaled[:, bpm_idx_train],
    y_train, y_test,
    "① BPM apenas (modelo original)"
)

# ── Passo 2: BPM + Passos ────────────────────────────────────────────────────
bpm_steps_features = bpm_only_features + ['steps_rolling_mean', 'is_active', 'bpm_excess']
bpm_steps_idx = [temporal_features.index(f) for f in bpm_steps_features]
ablation_eval(
    X_train_temp_scaled[:, bpm_steps_idx],
    X_test_temp_scaled[:, bpm_steps_idx],
    y_train, y_test,
    "② + Passos (steps, is_active, bpm_excess)"
)

# ── Passo 3: BPM + Passos + RR/HRV ──────────────────────────────────────────
bpm_steps_rr_features = bpm_steps_features + ['rr_rolling_mean', 'rr_rolling_std', 'rr_rmssd', 'rr_cv', 'irregular_rhythm']
bpm_steps_rr_idx = [temporal_features.index(f) for f in bpm_steps_rr_features]
ablation_eval(
    X_train_temp_scaled[:, bpm_steps_rr_idx],
    X_test_temp_scaled[:, bpm_steps_rr_idx],
    y_train, y_test,
    "③ + Ritmo Cardíaco / HRV (rr_*, irregular_rhythm)"
)

# ── Passo 4: todas as features temporais ────────────────────────────────────
ablation_eval(
    X_train_temp_scaled, X_test_temp_scaled,
    y_train, y_test,
    "④ Todas as features temporais"
)

# ── Passo 5: temporal + demográfico ─────────────────────────────────────────
ablation_eval(
    np.hstack([X_train_temp_scaled, X_train_demo_scaled]),
    np.hstack([X_test_temp_scaled,  X_test_demo_scaled]),
    y_train, y_test,
    "⑤ + Dados Demográficos (idade, sexo)"
)

# ── Passo 6: tudo (modelo final) ────────────────────────────────────────────
ablation_eval(
    X_train_flat, X_test_flat,
    y_train, y_test,
    "⑥ + Histórico Clínico  ← modelo final completo"
)

print("\nInterpretação:")
print("  - Recall-Normal  → % de casos saudáveis corretamente identificados (especificidade)")
print("  - Recall-Risco   → % de anomalias detectadas (sensibilidade clínica — mais crítico)")
print("  - F1-macro       → equilíbrio geral entre as duas classes")

In [ ]:
# Salva o modelo multi-input
model.save("models/cardiac_risk_model.keras")

# Salva os scalers (um para cada grupo de features contínuas)
joblib.dump(scaler_temporal, "models/scaler_temporal.pkl")
joblib.dump(scaler_demo, "models/scaler_demo.pkl")

# Salva as listas de features para uso em inferência
feature_config = {
    'temporal_features': temporal_features,
    'demographic_features': demographic_features,
    'clinical_features': clinical_features,
    'continuous_features': continuous_features,
    'binary_features': binary_features,
    'temporal_continuous_idx': temporal_continuous_idx,
    'demo_continuous_idx': demo_continuous_idx,
}
joblib.dump(feature_config, "models/feature_config.pkl")

print("Modelo e artefatos salvos:")
print("  - models/cardiac_risk_model.keras")
print("  - models/scaler_temporal.pkl")
print("  - models/scaler_demo.pkl")
print("  - models/feature_config.pkl")

['neuralNetwork/models/scaler.pkl']